# 第 13 课｜假设检验入门

> **数据身份**：固定随机种子生成的模拟教学数据，不是真实证券行情。  
> **环境**：`pkm-quant` · Python 3.11 · 默认离线运行。  
> **课程原始材料**：`13_youtube-fRHsVq24XRg/index.html`  
> **所属模块**：P1-M3 · 统计证据与可预测性

## 本节量化问题

策略平均收益显著不为零，是否足以证明策略可交易？

## 准确定义与策略报告位置

假设检验衡量样本与原假设的相容程度；p 值不是策略盈利概率，必须同时报告效应大小和区间。

- **核心关系**：t=(R̄-μ0)/(s/√n)；p 值是在原假设成立时得到当前或更极端统计量的概率。
- **写入报告**：原假设、效应大小、置信区间和检验功效
- **P2 衔接**：P2 第 48、56 课：统计检验与单元测试。


## 运行环境与数据契约

先确认项目、数据目录和数据身份。后续联网数据源必须通过适配层进入，不能改变本 Notebook 的研究计算口径。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 同时报告均值、区间和 p 值

检验原假设为平均日收益等于 0。95% 置信区间描述估计不确定性，不是未来单日收益区间。

In [2]:
from scipy import stats

daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
close = daily.pivot(index="date", columns="symbol", values="adj_close").sort_index()
strategy = close["SIM_VALUE"].pct_change(fill_method=None).dropna()

test = stats.ttest_1samp(strategy, popmean=0.0)
mean = strategy.mean()
se = stats.sem(strategy)
interval = stats.t.interval(0.95, df=len(strategy) - 1, loc=mean, scale=se)

display(pd.Series({
    "mean_daily_return": mean,
    "effect_bps_per_day": mean * 10_000,
    "ci95_low": interval[0],
    "ci95_high": interval[1],
    "pvalue": test.pvalue,
    "observations": len(strategy),
}))

mean_daily_return      -0.000085
effect_bps_per_day     -0.850286
ci95_low               -0.000744
ci95_high               0.000574
pvalue                  0.800149
observations          755.000000
dtype: float64

## 方法边界与阶段验收

多重检验、序列相关、非正态和选择偏差会放大假阳性。

完成本课后应能：

1. 不看课程材料准确解释核心概念和公式符号；
2. 用小样本手工核对 Notebook 的关键结果；
3. 说明结果能支持什么判断、不能支持什么判断；
4. 指出它在策略报告中的位置，以及仍缺少的数据、成本或稳健性证据。
